# 🧹 Data Cleaning & Sampling for Sentiment Analysis

**Goal:** Prepare clean data for IndoBERT sentiment analysis + BiLSTM time series  
**Input:** CNBC articles (raw)  
**Output:**
- Cleaned full dataset
- Random seed data (600-1000) for manual labeling
- Excel template for labeling

---

## 📦 Step 1: Setup

In [3]:
!pip install pandas numpy openpyxl
import pandas as pd
import numpy as np
import re
from datetime import datetime
import os

# Set random seed for reproducibility
np.random.seed(42)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Libraries loaded!")
print(f"Random seed: 42 (reproducible sampling)")

  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.0-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl (11.0 MB)
Using cached numpy-2.4.0-cp313-cp313-win_amd64.whl (12.3 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/6 [pytz]
   ---------------------------------------- 0/6 [pytz]
   ---------------------------------------- 0/6 [pytz]
   ---------------------------------------- 0/6 [pytz]
   ------ --------------------------------- 1/6 [tzdata]
   ------ --------------------------------- 1/6 [tzdata]
   ------ --------------------------------- 1/6 [tzdata]
   ------ --------------------------------- 1/6 [tzdata]
   ------ -------------------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Libraries loaded!
Random seed: 42 (reproducible sampling)


## 📥 Step 2: Load Data

In [7]:
print("="*70)
print("📥 LOADING DATA")
print("="*70)

# Load CNBC data
df = pd.read_csv('../data/data_berita/fixed/cnbc_articles.csv', encoding='utf-8-sig')
print(f"\n✅ Loaded: {len(df):,} articles")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst rows:")
display(df.head(3))

📥 LOADING DATA

✅ Loaded: 5,343 articles

Columns: ['url', 'title', 'content', 'content_length', 'scraped_at', 'date']

First rows:


,url,title,content,content_length,scraped_at,date
0,https://www.cnbcindonesia.com/news/20190930111242-8-103066/regulasi-investasi-rumit-ini-upaya-pe...,"Regulasi Investasi Rumit, Ini Upaya Pemerintah","Jakarta, CNBC Indonesia-Pemerintah terus mendorong peningkatan investasi melalui penerapan Omnib...",824,2026-01-05T06:52:59.562685,2019-09-30
1,https://www.cnbcindonesia.com/news/20190917101448-8-99934/menteri-arief-yahya-paparkan-strategi-...,Menteri Arief Yahya Paparkan Strategi Pariwisata RI,"Jakarta, CNBC Indonesia-Hingga akhir tahun 2018 jumlah wisatawan asing yang masuk ke Indonesia m...",759,2026-01-05T06:53:06.940368,2019-09-17
2,https://www.cnbcindonesia.com/news/20190904164843-4-97130/salip-ri-di-vietnam-tanah-gratis-buruh...,"Salip RI, di Vietnam Tanah Gratis, Buruhnya Produktif","Jakarta, CNBC Indonesia -Bank Dunia melaporkan dari aksi relokasi industri dari China ke negara-...",2274,2026-01-05T06:53:10.810281,2019-09-04


## 📅 Step 3: Date Parsing & Filtering

In [8]:
print("="*70)
print("📅 DATE PARSING & FILTERING")
print("="*70)

# Parse dates
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Remove rows with invalid dates
before = len(df)
df = df.dropna(subset=['date'])
print(f"\nRemoved {before - len(df)} rows with invalid dates")

# Filter 2019-2024 only
start_date = pd.to_datetime('2019-09-01')
end_date = pd.to_datetime('2024-09-30')

before = len(df)
df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
print(f"\nFiltered to 2019-09-01 to 2024-09-30:")
print(f"   Removed: {before - len(df)} articles outside range")
print(f"   Remaining: {len(df):,} articles")

print(f"\n📊 Date range:")
print(f"   Min: {df['date'].min().date()}")
print(f"   Max: {df['date'].max().date()}")

📅 DATE PARSING & FILTERING

Removed 0 rows with invalid dates

Filtered to 2019-09-01 to 2024-09-30:
   Removed: 0 articles outside range
   Remaining: 5,343 articles

📊 Date range:
   Min: 2019-09-01
   Max: 2024-09-30


## 🧹 Step 4: Text Cleaning

In [9]:
print("="*70)
print("🧹 TEXT CLEANING")
print("="*70)

def clean_text(text):
    """
    Clean text for sentiment analysis:
    - Remove URLs
    - Remove emails
    - Remove extra whitespace
    - Convert to lowercase (optional for IndoBERT)
    - Remove special characters except punctuation
    """
    if pd.isna(text) or text == "N/A":
        return ""
    
    text = str(text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove newlines and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning to title and content
print("\nCleaning titles...")
df['title_clean'] = df['title'].apply(clean_text)

print("Cleaning content...")
df['content_clean'] = df['content'].apply(clean_text)

# Remove rows with empty content after cleaning
before = len(df)
df = df[df['content_clean'].str.len() > 50]  # Minimum 50 chars
print(f"\n✅ Cleaned!")
print(f"   Removed {before - len(df)} articles with insufficient content")
print(f"   Remaining: {len(df):,} articles")

# Show sample
print(f"\n📋 Sample cleaned text:")
sample = df.iloc[0]
print(f"\nOriginal title: {sample['title'][:100]}")
print(f"Cleaned title: {sample['title_clean'][:100]}")
print(f"\nOriginal content (first 200 chars): {sample['content'][:200]}")
print(f"Cleaned content (first 200 chars): {sample['content_clean'][:200]}")

🧹 TEXT CLEANING

Cleaning titles...
Cleaning content...

✅ Cleaned!
   Removed 0 articles with insufficient content
   Remaining: 5,343 articles

📋 Sample cleaned text:

Original title: Regulasi Investasi Rumit, Ini Upaya Pemerintah
Cleaned title: Regulasi Investasi Rumit, Ini Upaya Pemerintah

Original content (first 200 chars): Jakarta, CNBC Indonesia-Pemerintah terus mendorong peningkatan investasi melalui penerapan Omnibus Law dengan memangkas dan menyederhanakan regulasi terkait izin usaha dan investasi yang dilaksanakan 
Cleaned content (first 200 chars): Jakarta, CNBC Indonesia-Pemerintah terus mendorong peningkatan investasi melalui penerapan Omnibus Law dengan memangkas dan menyederhanakan regulasi terkait izin usaha dan investasi yang dilaksanakan 


## 🔄 Step 5: Remove Duplicates

In [10]:
print("="*70)
print("🔄 REMOVING DUPLICATES")
print("="*70)

before = len(df)

# Remove exact title duplicates
df = df.drop_duplicates(subset=['title_clean'], keep='first')
after_title = len(df)
print(f"\n1. By title: Removed {before - after_title} duplicates")

# Remove similar content (first 100 chars)
df['content_hash'] = df['content_clean'].str[:100]
df = df.drop_duplicates(subset=['content_hash'], keep='first')
after_content = len(df)
print(f"2. By content: Removed {after_title - after_content} duplicates")

df = df.drop(columns=['content_hash'])

print(f"\n✅ Total removed: {before - after_content} duplicates")
print(f"   Final dataset: {len(df):,} unique articles")

🔄 REMOVING DUPLICATES

1. By title: Removed 15 duplicates
2. By content: Removed 268 duplicates

✅ Total removed: 283 duplicates
   Final dataset: 5,060 unique articles


## 📊 Step 6: Final Dataset Statistics

In [11]:
print("="*70)
print("📊 FINAL DATASET STATISTICS")
print("="*70)

print(f"\n✅ Total articles: {len(df):,}")
print(f"\n📅 Date range:")
print(f"   From: {df['date'].min().date()}")
print(f"   To: {df['date'].max().date()}")
print(f"   Days: {(df['date'].max() - df['date'].min()).days}")

# Monthly distribution
df['year_month'] = df['date'].dt.to_period('M')
monthly = df['year_month'].value_counts().sort_index()

print(f"\n📈 Monthly distribution:")
print(f"   Mean: {monthly.mean():.1f} articles/month")
print(f"   Median: {monthly.median():.0f} articles/month")
print(f"   Min: {monthly.min()} articles/month")
print(f"   Max: {monthly.max()} articles/month")

print(f"\n📋 First 6 months:")
for period, count in monthly.head(6).items():
    print(f"   {period}: {count:4d} articles")

print(f"\n📋 Last 6 months:")
for period, count in monthly.tail(6).items():
    print(f"   {period}: {count:4d} articles")

# Text length stats
df['content_length'] = df['content_clean'].str.len()
print(f"\n📏 Content length:")
print(f"   Mean: {df['content_length'].mean():.0f} chars")
print(f"   Median: {df['content_length'].median():.0f} chars")
print(f"   Min: {df['content_length'].min():.0f} chars")
print(f"   Max: {df['content_length'].max():.0f} chars")

📊 FINAL DATASET STATISTICS

✅ Total articles: 5,060

📅 Date range:
   From: 2019-09-01
   To: 2024-09-30
   Days: 1856

📈 Monthly distribution:
   Mean: 83.0 articles/month
   Median: 47 articles/month
   Min: 5 articles/month
   Max: 348 articles/month

📋 First 6 months:
   2019-09:   15 articles
   2019-10:   39 articles
   2019-11:   10 articles
   2019-12:    8 articles
   2020-01:    5 articles
   2020-02:   10 articles

📋 Last 6 months:
   2024-04:  171 articles
   2024-05:  179 articles
   2024-06:  202 articles
   2024-07:  220 articles
   2024-08:  348 articles
   2024-09:  168 articles

📏 Content length:
   Mean: 3163 chars
   Median: 2596 chars
   Min: 75 chars
   Max: 66713 chars


## 💾 Step 7: Save Cleaned Full Dataset

In [12]:
print("="*70)
print("💾 SAVING CLEANED FULL DATASET")
print("="*70)

# Select essential columns for sentiment analysis
df_clean = df[[
    'date',
    'title_clean',
    'content_clean',
    'content_length'
]].copy()

# Rename for clarity
df_clean.columns = ['date', 'title', 'content', 'content_length']

# Sort by date
df_clean = df_clean.sort_values('date').reset_index(drop=True)

# Save
output_file = 'cnbc_cleaned_full.csv'
df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')

file_size = os.path.getsize(output_file) / (1024**2)

print(f"\n✅ Saved: {output_file}")
print(f"   Rows: {len(df_clean):,}")
print(f"   Columns: {df_clean.columns.tolist()}")
print(f"   Size: {file_size:.2f} MB")

print(f"\n📋 Sample:")
display(df_clean.head(5))

💾 SAVING CLEANED FULL DATASET

✅ Saved: cnbc_cleaned_full.csv
   Rows: 5,060
   Columns: ['date', 'title', 'content', 'content_length']
   Size: 15.70 MB

📋 Sample:


,date,title,content,content_length
0,2019-09-01,"Kondusif, Menkominfo Siap Buka Kembali Internet di Papua","Jakarta, CNBC Indonesia -Menteri Koordinator Bidang Politik, Hukum, dan Keamanan (Menko Polhukam...",2063
1,2019-09-02,Wiranto Pastikan Papua Mulai Kondusif,"Jakarta, CNBC Indonesia-Pasca aksi demonstrasi yang berakhir ricuh di Papua dan Papua Barat, Men...",795
2,2019-09-03,"Brexit Berlarut-larut, Boris Johnson Ancam Parlemen Inggris","Jakarta, CNBC Indonesia- Perdana Menteri Inggris Boris Johnson secara implisit memperingatkan pa...",4047
3,2019-09-04,"Salip RI, di Vietnam Tanah Gratis, Buruhnya Produktif","Jakarta, CNBC Indonesia -Bank Dunia melaporkan dari aksi relokasi industri dari China ke negara-...",2274
4,2019-09-04,Pemimpin Hong Kong Resmi Cabut RUU Ekstradisi,"Jakarta, CNBC Indonesia -Kepala Eksekutif Hong Kong, Carrie Lam akhirnya secara resmi menarik ra...",453


## 🎲 Step 8: Random Sampling for Manual Labeling (Seed=42)

In [13]:
print("="*70)
print("🎲 RANDOM SAMPLING FOR MANUAL LABELING")
print("="*70)

# Set sample size (600-1000)
SAMPLE_SIZE = 800  # Adjust this: 600, 700, 800, 900, or 1000

print(f"\nSample size: {SAMPLE_SIZE}")
print(f"Random seed: 42 (reproducible)")

# Random sampling with seed=42
df_sample = df_clean.sample(n=SAMPLE_SIZE, random_state=42)
df_sample = df_sample.sort_values('date').reset_index(drop=True)

print(f"\n✅ Sampled {len(df_sample):,} articles")

# Check distribution
print(f"\n📊 Sample distribution:")
sample_monthly = df_sample.groupby(df_sample['date'].dt.to_period('M')).size()
print(f"   Months covered: {len(sample_monthly)}")
print(f"   Mean per month: {sample_monthly.mean():.1f}")

print(f"\n📅 Date range:")
print(f"   From: {df_sample['date'].min().date()}")
print(f"   To: {df_sample['date'].max().date()}")

🎲 RANDOM SAMPLING FOR MANUAL LABELING

Sample size: 800
Random seed: 42 (reproducible)

✅ Sampled 800 articles

📊 Sample distribution:
   Months covered: 61
   Mean per month: 13.1

📅 Date range:
   From: 2019-09-11
   To: 2024-09-30


## 📝 Step 9: Prepare for Manual Labeling

In [14]:
print("="*70)
print("📝 PREPARING FOR MANUAL LABELING")
print("="*70)

# Create labeling dataframe
df_labeling = df_sample.copy()

# Add ID column
df_labeling.insert(0, 'id', range(1, len(df_labeling) + 1))

# Add text preview (first 300 chars for quick reading)
df_labeling['text_preview'] = df_labeling['content'].str[:300] + '...'

# Add empty sentiment column (to be filled manually)
df_labeling['sentiment'] = ''

# Add confidence column (optional)
df_labeling['confidence'] = ''

# Add notes column (optional)
df_labeling['notes'] = ''

# Reorder columns for easy labeling
df_labeling = df_labeling[[
    'id',
    'date',
    'title',
    'text_preview',
    'sentiment',      # ← FILL THIS!
    'confidence',     # Optional
    'notes',          # Optional
    'content',        # Full text (for reference)
    'content_length'
]]

print(f"\n✅ Prepared labeling dataset")
print(f"   Rows: {len(df_labeling):,}")
print(f"   Columns: {df_labeling.columns.tolist()}")

print(f"\n📋 Sample for labeling:")
display(df_labeling[['id', 'date', 'title', 'text_preview', 'sentiment']].head(5))

📝 PREPARING FOR MANUAL LABELING

✅ Prepared labeling dataset
   Rows: 800
   Columns: ['id', 'date', 'title', 'text_preview', 'sentiment', 'confidence', 'notes', 'content', 'content_length']

📋 Sample for labeling:


,id,date,title,text_preview,sentiment
0,1,2019-09-11,Menko Luhut : Jokowi Tak Mungkin Asal Pilih Menteri!,"Jakarta, CNBC Indonesia- Presiden Joko Widodo (Jokowi) menegaskan susuan kabinet baru sudah fina...",
1,2,2019-09-24,Ini Siasat Lobi-Lobi Boris Saat Brexit Makin Ujung Tanduk,"Jakarta, CNBC Indonesia-Kurang dari 40 hari jelang tenggat waktu Inggris keluar dari Uni Eropa, ...",
2,3,2019-10-06,"Menterinya Terbanyak Se-ASEAN, Kabinet RI Bakal Kian Gemuk","CNBC Indonesia, Jakarta -Jelang pelantikan Kabinet Kerja Periode II pada 20 Oktober, publik mena...",
3,4,2019-10-16,Akankah Gerindra Masuk Koalisi Jokowi?,"Jakarta, CNBC Indonesia -Partai Gerindra menggelar rapat kerja nasional di kediaman Ketua Umumny...",
4,5,2019-10-16,"Kubu Prabowo Masuk Koalisi Jokowi, Berkah atau Musibah?","Berbicara mengenai perekonomian, investasi menjadi elemen yang sangat krusial. Kalau berbicara m...",


## 💾 Step 10: Save Files for Labeling

In [15]:
print("="*70)
print("💾 SAVING LABELING FILES")
print("="*70)

# 1. Save CSV for labeling
labeling_file = f'seed_for_labeling_{SAMPLE_SIZE}.csv'
df_labeling.to_csv(labeling_file, index=False, encoding='utf-8-sig')

file_size = os.path.getsize(labeling_file) / (1024**2)
print(f"\n1️⃣ CSV for labeling:")
print(f"   File: {labeling_file}")
print(f"   Size: {file_size:.2f} MB")
print(f"   ℹ️ Open in Excel and fill 'sentiment' column")

# 2. Save Excel with instructions
excel_file = f'seed_for_labeling_{SAMPLE_SIZE}.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Main labeling sheet
    df_labeling.to_excel(writer, sheet_name='Labeling', index=False)
    
    # Guidelines sheet
    guidelines = pd.DataFrame({
        'Category': ['POSITIVE', 'NEGATIVE', 'NEUTRAL'],
        'Description': [
            'Berita positif: ekonomi naik, kebijakan sukses, investasi masuk, stabilitas',
            'Berita negatif: krisis, konflik, korupsi, ekonomi turun, demo/protes',
            'Berita netral: pengumuman rutin, laporan, meeting, tidak ada sentimen jelas'
        ],
        'Keywords': [
            'naik, tumbuh, sukses, optimis, bagus, untung, stabil, meningkat',
            'turun, krisis, gagal, konflik, demo, korupsi, menurun, jatuh',
            'umumkan, laporkan, adakan, hadiri, bahas, diskusi, rapat'
        ],
        'Examples': [
            'Ekonomi Indonesia Tumbuh 5.2%, Investasi Asing Meningkat',
            'Rupiah Melemah, Demo Tolak Omnibus Law, Menteri Korupsi',
            'Presiden Hadiri Rapat Kabinet, Menteri Laporkan Kinerja'
        ]
    })
    guidelines.to_excel(writer, sheet_name='Guidelines', index=False)

file_size = os.path.getsize(excel_file) / (1024**2)
print(f"\n2️⃣ Excel with guidelines:")
print(f"   File: {excel_file}")
print(f"   Size: {file_size:.2f} MB")
print(f"   ℹ️ Open in Excel, see 'Guidelines' sheet for instructions")

print(f"\n{'='*70}")
print(f"✅ ALL FILES SAVED!")
print(f"{'='*70}")

💾 SAVING LABELING FILES

1️⃣ CSV for labeling:
   File: seed_for_labeling_800.csv
   Size: 2.70 MB
   ℹ️ Open in Excel and fill 'sentiment' column

2️⃣ Excel with guidelines:
   File: seed_for_labeling_800.xlsx
   Size: 0.87 MB
   ℹ️ Open in Excel, see 'Guidelines' sheet for instructions

✅ ALL FILES SAVED!


## 📊 Step 11: Save Remaining Unlabeled Data

In [16]:
print("="*70)
print("📊 SAVING REMAINING UNLABELED DATA")
print("="*70)

# Get remaining data (not in sample)
sampled_ids = df_sample.index
df_remaining = df_clean.loc[~df_clean.index.isin(sampled_ids)].copy()
df_remaining = df_remaining.reset_index(drop=True)

# Save
remaining_file = 'cnbc_remaining_for_prediction.csv'
df_remaining.to_csv(remaining_file, index=False, encoding='utf-8-sig')

file_size = os.path.getsize(remaining_file) / (1024**2)

print(f"\n✅ Saved: {remaining_file}")
print(f"   Rows: {len(df_remaining):,}")
print(f"   Size: {file_size:.2f} MB")
print(f"   ℹ️ This will be predicted after training on labeled seed")

📊 SAVING REMAINING UNLABELED DATA

✅ Saved: cnbc_remaining_for_prediction.csv
   Rows: 4,260
   Size: 13.55 MB
   ℹ️ This will be predicted after training on labeled seed


## 📋 SUMMARY

In [17]:
print("="*70)
print("📋 FINAL SUMMARY")
print("="*70)

print(f"\n✅ COMPLETED SUCCESSFULLY!\n")

print(f"📊 Dataset Summary:")
print(f"   Total cleaned articles: {len(df_clean):,}")
print(f"   Period: {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")

print(f"\n📁 Files Created:")
print(f"\n1️⃣ cnbc_cleaned_full.csv")
print(f"   • Full cleaned dataset ({len(df_clean):,} articles)")
print(f"   • Ready for sentiment analysis & time series")
print(f"   • Columns: date, title, content, content_length")

print(f"\n2️⃣ seed_for_labeling_{SAMPLE_SIZE}.csv")
print(f"   • Random sample ({SAMPLE_SIZE} articles, seed=42)")
print(f"   • For manual labeling")
print(f"   • Fill 'sentiment' column: positive/negative/neutral")

print(f"\n3️⃣ seed_for_labeling_{SAMPLE_SIZE}.xlsx")
print(f"   • Excel version with guidelines sheet")
print(f"   • Easier for manual labeling")
print(f"   • Includes labeling instructions")

print(f"\n4️⃣ cnbc_remaining_for_prediction.csv")
print(f"   • Remaining unlabeled data ({len(df_remaining):,} articles)")
print(f"   • Will be predicted after training")

print(f"\n{'='*70}")
print(f"🎯 NEXT STEPS:")
print(f"{'='*70}")
print(f"\n1. Open: seed_for_labeling_{SAMPLE_SIZE}.xlsx")
print(f"2. Read: Guidelines sheet for labeling instructions")
print(f"3. Label: Fill 'sentiment' column for {SAMPLE_SIZE} articles")
print(f"4. Save: As seed_for_labeling_{SAMPLE_SIZE}_LABELED.xlsx")
print(f"5. Train: Use labeled data to train IndoBERT")
print(f"6. Predict: Apply model to remaining {len(df_remaining):,} articles")
print(f"\n{'='*70}")
print(f"✨ Random seed: 42 (reproducible sampling)")
print(f"✨ Ready for sentiment analysis!")
print(f"{'='*70}")

📋 FINAL SUMMARY

✅ COMPLETED SUCCESSFULLY!

📊 Dataset Summary:
   Total cleaned articles: 5,060
   Period: 2019-09-01 to 2024-09-30

📁 Files Created:

1️⃣ cnbc_cleaned_full.csv
   • Full cleaned dataset (5,060 articles)
   • Ready for sentiment analysis & time series
   • Columns: date, title, content, content_length

2️⃣ seed_for_labeling_800.csv
   • Random sample (800 articles, seed=42)
   • For manual labeling
   • Fill 'sentiment' column: positive/negative/neutral

3️⃣ seed_for_labeling_800.xlsx
   • Excel version with guidelines sheet
   • Easier for manual labeling
   • Includes labeling instructions

4️⃣ cnbc_remaining_for_prediction.csv
   • Remaining unlabeled data (4,260 articles)
   • Will be predicted after training

🎯 NEXT STEPS:

1. Open: seed_for_labeling_800.xlsx
2. Read: Guidelines sheet for labeling instructions
3. Label: Fill 'sentiment' column for 800 articles
4. Save: As seed_for_labeling_800_LABELED.xlsx
5. Train: Use labeled data to train IndoBERT
6. Predict: Ap